# BiLSTM + Attention para Detección DGA
**Referencia:** Namgung et al., *Efficient Deep Learning Models for DGA Domain Detection*, Security and Communication Networks 2021.

**Arquitectura:**
- Character-level embedding (input_dim=75 → output_dim=32)
- BiLSTM bidireccional: 128 nodos por dirección (256 total), return_sequences=True
- Self-Attention sobre la secuencia de salida BiLSTM
- Fully Connected: ReLU + Dropout(0.5) → sigmoid
- Clasificación binaria: DGA vs legit

**Entrenamiento:** train_1M.csv | **Evaluación:** 54 familias × 30 runs (archivos pre-generados)

In [1]:
# ── Verificar GPU ──────────────────────────────────────────────────────────────
import torch
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

CUDA disponible: True
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN — Ajusta estas rutas antes de correr
# ══════════════════════════════════════════════════════════════════════════════
from pathlib import Path

DATASET_PATH      = Path('/content/drive/MyDrive/NEW_DGA_DETECTOR/train_1M.csv')
OUTPUT_DIR        = Path('/content/drive/MyDrive/NEW_DGA_DETECTOR/models/bilstm-run-001')
FAMILIAS_TEST_DIR = '/content/drive/My Drive/Familias_Test'  # misma carpeta que usa el CNN
RESULTS_DIR       = Path('/content/drive/MyDrive/NEW_DGA_DETECTOR/results/bilstm')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Hiperparámetros (Namgung et al. 2021)
MAXLEN       = 75     # longitud máxima (paper: 73; proyecto usa 75)
EMBED_DIM    = 32     # paper: matrix 73×32
BILSTM_DIM   = 128    # por dirección → 256 total (paper: 128)
DROPOUT      = 0.5    # paper: 0.5
FC_HIDDEN    = 64     # capa FC intermedia
BATCH_SIZE   = 256    # reducido para memoria T4 con BiLSTM
EPOCHS       = 10     # paper: 10
LR           = 1e-4   # paper: Adam lr=1e-4
TEST_SIZE    = 0.2
RANDOM_STATE = 42
RUNS         = 30

print('Config OK')
print(f'Dataset:       {DATASET_PATH}')
print(f'Output:        {OUTPUT_DIR}')
print(f'Test familias: {FAMILIAS_TEST_DIR}')

Config OK
Dataset:       /content/drive/MyDrive/NEW_DGA_DETECTOR/train_1M.csv
Output:        /content/drive/MyDrive/NEW_DGA_DETECTOR/models/bilstm-run-001
Test familias: /content/drive/My Drive/Familias_Test


In [9]:
# ── Imports ────────────────────────────────────────────────────────────────────
import re
import json
import time
import string
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, confusion_matrix, classification_report)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cuda


In [10]:
# ── Tokenizador de caracteres (consistente con proyecto) ───────────────────────
# Namgung usa: a-z (0-25), 0-9 (26-35), '.', '-', '_' (36-38) → total 39 tokens
# Aquí usamos la misma convención que el resto del proyecto
CHARS    = string.ascii_lowercase + string.digits + '-._'
CHAR2IDX = {c: i + 1 for i, c in enumerate(CHARS)}  # 0 = padding
VOCAB_SIZE = len(CHARS) + 1  # 40

def encode_domain(domain: str) -> list:
    domain = str(domain).lower().strip()
    encoded = [CHAR2IDX.get(c, 0) for c in domain[:MAXLEN]]
    # Right-padding con ceros (Namgung usa zero-padding al final)
    encoded += [0] * (MAXLEN - len(encoded))
    return encoded

test_enc = encode_domain('google.com')
print(f'Vocab size: {VOCAB_SIZE}')
print(f'Encoded "google.com" (primeros 15): {test_enc[:15]}')

Vocab size: 40
Encoded "google.com" (primeros 15): [7, 15, 15, 7, 12, 5, 38, 3, 15, 13, 0, 0, 0, 0, 0]


In [11]:
# ── Carga y limpieza de datos (idéntico al notebook Logit) ────────────────────
DOMAIN_RE = re.compile(r'^[a-z0-9.-]+$')

def normalize_domain(value: str) -> str:
    from urllib.parse import urlparse
    domain = str(value).strip().lower()
    parsed = urlparse(domain)
    if parsed.netloc:
        domain = parsed.netloc
    domain = domain.split('@')[-1].split('/')[0].split(':')[0].rstrip('.')
    domain = re.sub(r'\s+', '', domain)
    if not DOMAIN_RE.match(domain):
        domain = re.sub(r'[^a-z0-9.-]', '', domain)
    return domain

t0 = time.time()
df_raw = pd.read_csv(DATASET_PATH)
print(f'Filas brutas: {len(df_raw):,}  ({time.time()-t0:.1f}s)')

df = pd.DataFrame()
df['domain'] = df_raw['domain'].map(normalize_domain)
df['label']  = (df_raw['label'].str.strip().str.lower() == 'dga').astype(int)
df['family'] = df_raw['family'].astype(str).str.strip().str.lower()
df = df[df['domain'].ne('')].drop_duplicates(subset=['domain', 'label']).reset_index(drop=True)

print(f'Filas limpias: {len(df):,}')
print(f'Labels: {df["label"].value_counts().to_dict()}')
print(f'Familias: {df["family"].nunique()}')
df.head()

Filas brutas: 1,080,000  (1.8s)
Filas limpias: 1,053,080
Labels: {0: 540000, 1: 513080}
Familias: 55


,domain,label,family
0,newmedialove.ru,0,legit
1,bankesb.com,0,legit
2,sbjnvufhillsszger.net,1,fobber
3,rpltm.online,0,legit
4,theblotsays.com,0,legit


In [12]:
# ── Split honesto por familia (idéntico al notebook Logit/ModernBERT) ──────────
dga_df    = df[df['label'] == 1].reset_index(drop=True)
benign_df = df[df['label'] == 0].reset_index(drop=True)

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
dga_train_idx, dga_test_idx = next(
    splitter.split(dga_df['domain'], dga_df['label'], groups=dga_df['family'])
)
dga_train = dga_df.iloc[dga_train_idx]
dga_test  = dga_df.iloc[dga_test_idx]

b_train_idx, b_test_idx = train_test_split(
    benign_df.index, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
benign_train = benign_df.loc[b_train_idx]
benign_test  = benign_df.loc[b_test_idx]

train_df = pd.concat([dga_train, benign_train]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
test_df  = pd.concat([dga_test,  benign_test ]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'Train: {len(train_df):,} | Test: {len(test_df):,}')
print(f'Train labels: {train_df["label"].value_counts().to_dict()}')
print(f'Train familias DGA: {train_df[train_df["label"]==1]["family"].nunique()}')
print(f'Test  familias DGA: {test_df[test_df["label"]==1]["family"].nunique()}')

Train: 845,639 | Test: 207,441
Train labels: {0: 432000, 1: 413639}
Train familias DGA: 43
Test  familias DGA: 11


In [13]:
# ── PyTorch Dataset & DataLoader ───────────────────────────────────────────────
class DGADataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.X = torch.tensor(
            [encode_domain(d) for d in df['domain']], dtype=torch.long
        )
        self.y = torch.tensor(df['label'].values, dtype=torch.float32)

    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_ds = DGADataset(train_df)
test_ds  = DGADataset(test_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Batches train: {len(train_loader)} | Batches test: {len(test_loader)}')

Batches train: 3304 | Batches test: 811


In [14]:
# ── Modelo BiLSTM + Self-Attention (Namgung et al. 2021) ──────────────────────
class SelfAttention(nn.Module):
    """Self-attention sobre la secuencia de salida del BiLSTM.
    Aprende qué posiciones del dominio son más discriminativas."""
    def __init__(self, hidden_size: int):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, lstm_out):
        # lstm_out: (B, L, H)
        scores  = self.attn(lstm_out)          # (B, L, 1)
        weights = torch.softmax(scores, dim=1) # (B, L, 1)
        context = (weights * lstm_out).sum(dim=1)  # (B, H)
        return context


class BiLSTMAttention(nn.Module):
    """
    Arquitectura Namgung 2021:
    - Embedding de caracteres (VOCAB × EMBED_DIM)
    - BiLSTM(128 por dirección → 256 total), return_sequences=True
    - Self-Attention → vector de contexto (256)
    - FC: 256 → FC_HIDDEN → ReLU → Dropout(0.5) → 1 → sigmoid
    """
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)

        self.bilstm = nn.LSTM(
            input_size=EMBED_DIM,
            hidden_size=BILSTM_DIM,
            batch_first=True,
            bidirectional=True
        )

        bilstm_out = BILSTM_DIM * 2   # 256
        self.attention = SelfAttention(bilstm_out)

        self.fc = nn.Sequential(
            nn.Linear(bilstm_out, FC_HIDDEN),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(FC_HIDDEN, 1)
        )

    def forward(self, x):
        emb = self.embedding(x)          # (B, L, EMBED_DIM)
        out, _ = self.bilstm(emb)        # (B, L, 2*BILSTM_DIM)
        context = self.attention(out)    # (B, 2*BILSTM_DIM)
        return self.fc(context).squeeze(1)  # (B,)


model = BiLSTMAttention().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parámetros entrenables: {total_params:,}')
print(model)

Parámetros entrenables: 183,937
BiLSTMAttention(
  (embedding): Embedding(40, 32, padding_idx=0)
  (bilstm): LSTM(32, 128, batch_first=True, bidirectional=True)
  (attention): SelfAttention(
    (attn): Linear(in_features=256, out_features=1, bias=False)
  )
  (fc): Sequential(
    (0): Linear(in_features=256, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [15]:
# ── Entrenamiento ──────────────────────────────────────────────────────────────
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scaler    = GradScaler()

best_val_f1 = 0.0
history = []

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    t_epoch = time.time()

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        with autocast():
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ── Validación ─────────────────────────────────────────────────────────────
    model.eval()
    all_preds, all_labels = [], []
    val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)
            with autocast():
                logits = model(X_batch)
                loss   = criterion(logits, y_batch)
            val_loss += loss.item()
            preds = (torch.sigmoid(logits) >= 0.5).long().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y_batch.long().cpu().numpy())

    val_loss /= len(test_loader)
    val_f1  = f1_score(all_labels, all_preds, zero_division=0)
    val_acc = accuracy_score(all_labels, all_preds)
    elapsed = time.time() - t_epoch

    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | '
          f'val_f1={val_f1:.4f} | val_acc={val_acc:.4f} | {elapsed:.1f}s')

    history.append({'epoch': epoch, 'train_loss': train_loss,
                    'val_loss': val_loss, 'val_f1': val_f1, 'val_acc': val_acc})

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), OUTPUT_DIR / 'bilstm_best.pth')
        print(f'  ✓ Mejor modelo guardado (F1={best_val_f1:.4f})')

print(f'\nEntrenamiento completado. Mejor val_F1={best_val_f1:.4f}')

/tmp/ipykernel_1369/1573436641.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()
/tmp/ipykernel_1369/1573436641.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1369/1573436641.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 01/10 | train_loss=0.3427 | val_loss=0.3020 | val_f1=0.8717 | val_acc=0.8852 | 35.3s
  ✓ Mejor modelo guardado (F1=0.8717)


/tmp/ipykernel_1369/1573436641.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1369/1573436641.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 02/10 | train_loss=0.2533 | val_loss=0.3082 | val_f1=0.8748 | val_acc=0.8859 | 33.5s
  ✓ Mejor modelo guardado (F1=0.8748)


/tmp/ipykernel_1369/1573436641.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1369/1573436641.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


...

Epoch 08/10 | train_loss=0.1784 | val_loss=0.3398 | val_f1=0.8764 | val_acc=0.8894 | 32.8s


/tmp/ipykernel_1369/1573436641.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1369/1573436641.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 09/10 | train_loss=0.1721 | val_loss=0.3683 | val_f1=0.8727 | val_acc=0.8857 | 32.3s


/tmp/ipykernel_1369/1573436641.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1369/1573436641.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 10/10 | train_loss=0.1665 | val_loss=0.3177 | val_f1=0.8739 | val_acc=0.8835 | 33.0s

Entrenamiento completado. Mejor val_F1=0.8822


In [16]:
# ── Guardar modelo final + reporte ────────────────────────────────────────────
model.load_state_dict(torch.load(OUTPUT_DIR / 'bilstm_best.pth', map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch.to(DEVICE))
        preds  = (torch.sigmoid(logits) >= 0.5).long().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.long().numpy())

report = classification_report(all_labels, all_preds, digits=4)
metrics = {
    'model': 'BiLSTM+Attention',
    'reference': 'Namgung et al., Security and Communication Networks 2021',
    'train_rows': len(train_df),
    'test_rows': len(test_df),
    'best_val_f1': best_val_f1,
    'final_f1': float(f1_score(all_labels, all_preds, zero_division=0)),
    'final_precision': float(precision_score(all_labels, all_preds, zero_division=0)),
    'final_recall': float(recall_score(all_labels, all_preds, zero_division=0)),
    'final_accuracy': float(accuracy_score(all_labels, all_preds)),
    'history': history
}

(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
(OUTPUT_DIR / 'classification_report.txt').write_text(report)

print(report)
print(f'Guardado en: {OUTPUT_DIR}')

              precision    recall  f1-score   support

           0     0.8600    0.9502    0.9028    108000
           1     0.9389    0.8320    0.8822     99441

    accuracy                         0.8935    207441
   macro avg     0.8995    0.8911    0.8925    207441
weighted avg     0.8978    0.8935    0.8930    207441

Guardado en: /content/drive/MyDrive/NEW_DGA_DETECTOR/models/bilstm-run-001


In [17]:
# ── Helpers para evaluación por familia ───────────────────────────────────────
def predict_domains(domains_series: pd.Series) -> np.ndarray:
    """Recibe una Serie de dominios, devuelve predicciones binarias."""
    model.eval()
    encoded = torch.tensor(
        [encode_domain(d) for d in domains_series], dtype=torch.long
    )
    all_preds = []
    with torch.no_grad():
        for i in range(0, len(encoded), 256):
            batch  = encoded[i:i+256].to(DEVICE)
            logits = model(batch)
            preds  = (torch.sigmoid(logits) >= 0.5).long().cpu().numpy()
            all_preds.extend(preds)
    return np.array(all_preds)

def fpr_tpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return fpr, tpr

print('Helpers cargados')

Helpers cargados


In [18]:
# ── Evaluación por familia (54 familias × 30 runs) ────────────────────────────
# Mismo protocolo que CNN: 50 DGA + 50 legit por run, leyendo de Familias_Test
FAMILIES = [
    'alureon.gz','bamital.gz','banjori.gz','bedep.gz','charbot.gz','chinad.gz',
    'conficker.gz','corebot.gz','cryptolocker.gz','deception.gz','dircrypt.gz',
    'dnschanger.gz','dyre.gz','emotet.gz','fobber.gz','gameover.gz','gozi.gz',
    'kraken.gz','locky.gz','manuelita.gz','matsnu.gz','monerominer.gz','murofet.gz',
    'murofetweekly.gz','mydoom.gz','necurs.gz','nymaim.gz','oderoor.gz','padcrypt.gz',
    'pitou.gz','proslikefan.gz','pushdo.gz','pykspa.gz','qadars.gz','qakbot.gz',
    'qsnatch.gz','ramdo.gz','ramnit.gz','ranbyus.gz','rovnix.gz','shiotob.gz',
    'simda.gz','sisron.gz','sphinx.gz','suppobox.gz','symmi.gz','tempedreve.gz',
    'tinba.gz','tinynuke.gz','vawtrak.gz','vidro.gz','virut.gz','zeus-newgoz.gz',
    'zloader.gz'
]

results = []

for family in FAMILIES:
    acc_l, pre_l, rec_l, f1_l = [], [], [], []
    fpr_l, tpr_l, qt_l = [], [], []

    # Un reader por familia — chunksize=50 → run 0 lee filas 0-49, run 1 lee 50-99, etc.
    # Igual que CNN: los mismos dominios en cada run para comparación justa
    try:
        dga_reader   = pd.read_csv(f'{FAMILIAS_TEST_DIR}/{family}', chunksize=50)
        legit_reader = pd.read_csv(f'{FAMILIAS_TEST_DIR}/legit.gz', chunksize=50)
    except FileNotFoundError as e:
        print(f'[WARN] Archivo no encontrado: {e}')
        continue

    for run in range(RUNS):
        try:
            dga_chunk   = dga_reader.get_chunk()
            legit_chunk = legit_reader.get_chunk()
        except StopIteration:
            print(f'[WARN] {family}: sin suficientes chunks en run {run}')
            break

        df_run = pd.concat([dga_chunk, legit_chunk], ignore_index=True)
        y_true = (df_run['label'] == 'dga').astype(int).values

        t0 = time.perf_counter()
        y_pred = predict_domains(df_run['domain'])
        elapsed = time.perf_counter() - t0
        per_domain_s = elapsed / len(df_run)

        acc_l.append(accuracy_score(y_true, y_pred))
        pre_l.append(precision_score(y_true, y_pred, zero_division=0))
        rec_l.append(recall_score(y_true, y_pred, zero_division=0))
        f1_l.append(f1_score(y_true, y_pred, zero_division=0))
        fpr_v, tpr_v = fpr_tpr(y_true, y_pred)
        fpr_l.append(fpr_v)
        tpr_l.append(tpr_v)
        qt_l.append(per_domain_s)

    name = family.split('.')[0]
    print(f'{name:20}: acc={np.mean(acc_l):.3f} f1={np.mean(f1_l):.3f} '
          f'pre={np.mean(pre_l):.3f} rec={np.mean(rec_l):.3f} '
          f'FPR={np.mean(fpr_l):.3f} TPR={np.mean(tpr_l):.3f}')

    results.append({
        'Family':            name,
        'Accuracy Mean':     np.mean(acc_l),   'Accuracy Std Dev':   np.std(acc_l),
        'F1 Score Mean':     np.mean(f1_l),    'F1 Score Std Dev':   np.std(f1_l),
        'Precision Mean':    np.mean(pre_l),   'Precision Std Dev':  np.std(pre_l),
        'Recall Mean':       np.mean(rec_l),   'Recall Std Dev':     np.std(rec_l),
        'FPR Mean':          np.mean(fpr_l),   'FPR Std Dev':        np.std(fpr_l),
        'TPR Mean':          np.mean(tpr_l),   'TPR Std Dev':        np.std(tpr_l),
        'Query Time Mean':   np.mean(qt_l),    'Query Time Std Dev': np.std(qt_l),
    })

print('\nEvaluación completada.')

alureon             : acc=0.942 f1=0.942 pre=0.941 rec=0.945 FPR=0.060 TPR=0.945
bamital             : acc=0.970 f1=0.971 pre=0.944 rec=1.000 FPR=0.060 TPR=1.000
banjori             : acc=0.690 f1=0.585 pre=0.884 rec=0.440 FPR=0.060 TPR=0.440
bedep               : acc=0.965 f1=0.966 pre=0.944 rec=0.989 FPR=0.060 TPR=0.989
charbot             : acc=0.618 f1=0.434 pre=0.832 rec=0.297 FPR=0.060 TPR=0.297
chinad              : acc=0.970 f1=0.971 pre=0.944 rec=0.999 FPR=0.060 TPR=0.999
conficker           : acc=0.886 f1=0.879 pre=0.934 rec=0.832 FPR=0.060 TPR=0.832
corebot             : acc=0.970 f1=0.971 pre=0.944 rec=1.000 FPR=0.060 TPR=1.000
cryptolocker        : acc=0.952 f1=0.953 pre=0.942 rec=0.965 FPR=0.060 TPR=0.965
deception           : acc=0.965 f1=0.966 pre=0.944 rec=0.989 FPR=0.060 TPR=0.989
dircrypt            : acc=0.957 f1=0.958 pre=0.943 rec=0.975 FPR=0.060 TPR=0.975
dnschanger          : acc=0.941 f1=0.941 pre=0.941 rec=0.941 FPR=0.060 TPR=0.941
dyre                : acc=0.

In [19]:
# ── Guardar resultados y promedios finales ─────────────────────────────────────
df_results = pd.DataFrame(results)

results_csv = RESULTS_DIR / 'df_results_BiLSTM.csv'
df_results.to_csv(results_csv, index=False)
print(f'Resultados guardados: {results_csv}')

avg_cols = ['Accuracy Mean','F1 Score Mean','Precision Mean','Recall Mean',
            'FPR Mean','TPR Mean','Query Time Mean']
avg = df_results[avg_cols].mean().to_dict()

final_metrics = {
    'model': 'BiLSTM+Attention',
    'reference': 'Namgung et al., Security and Communication Networks 2021',
    **avg
}
final_json = RESULTS_DIR / 'final_test_metrics_BiLSTM.json'
final_json.write_text(json.dumps(final_metrics, indent=2))

print('\n── Métricas promedio (54 familias) ──')
for k, v in avg.items():
    print(f'  {k}: {v:.4f}')

display(df_results)

Resultados guardados: /content/drive/MyDrive/NEW_DGA_DETECTOR/results/bilstm/df_results_BiLSTM.csv

── Métricas promedio (54 familias) ──
  Accuracy Mean: 0.8916
  F1 Score Mean: 0.8556
  Precision Mean: 0.9134
  Recall Mean: 0.8433
  FPR Mean: 0.0600
  TPR Mean: 0.8433
  Query Time Mean: 0.0001


,Family,Accuracy Mean,Accuracy Std Dev,F1 Score Mean,F1 Score Std Dev,Precision Mean,Precision Std Dev,Recall Mean,Recall Std Dev,FPR Mean,FPR Std Dev,TPR Mean,TPR Std Dev,Query Time Mean,Query Time Std Dev
0,alureon,0.942333,0.025779,0.942467,0.025904,0.940960,0.031656,0.944667,0.031277,0.06,0.03266,0.944667,0.031277,0.000064,0.000003
1,bamital,0.970000,0.016330,0.971120,0.015514,0.944306,0.029545,1.000000,0.000000,0.06,0.03266,1.000000,0.000000,0.000065,0.000002
2,banjori,0.690000,0.030000,0.584808,0.052731,0.883911,0.060911,0.440000,0.056332,0.06,0.03266,0.440000,0.056332,0.000065,0.000003
3,bedep,0.964667,0.018927,0.965714,0.018261,0.943666,0.029863,0.989333,0.014360,0.06,0.03266,0.989333,0.014360,0.000064,0.000003
4,charbot,0.618333,0.038651,0.433957,0.079221,0.832171,0.089848,0.296667,0.066349,0.06,0.03266,0.296667,0.066349,0.000065,0.000003
5,chinad,0.969667,0.016428,0.970783,0.015624,0.944270,0.029546,0.999333,0.003590,0.06,0.03266,0.999333,0.003590,0.000064,0.000002
6,conficker,0.886000,0.034409,0.878595,0.039248,0.933718,0.035123,0.832000,0.060575,0.06,0.03266,0.832000,0.060575,0.000074,0.000004
7,corebot,0.970000,0.016330,0.971120,0.015514,0.944306,0.029545,1.000000,0.000000,0.06,0.03266,1.000000,0.000000,0.000073,0.000002
8,cryptolocker,0.952333,0.022164,0.953004,0.021865,0.942257,0.030632,0.964667,0.025131,0.06,0.03266,0.964667,0.025131,0.000071,0.000006
9,deception,0.964667,0.018927,0.965707,0.018248,0.943682,0.029922,0.989333,0.015261,0.06,0.03266,0.989333,0.015261,0.000065,0.000003


In [ ]:
# ── Descargar archivos localmente ─────────────────────────────────────────────
import shutil
from google.colab import files

for src in [results_csv, final_json, OUTPUT_DIR / 'metrics.json']:
    dst = Path('/content') / src.name
    shutil.copy(src, dst)
    files.download(str(dst))
    print(f'Descargando: {src.name}')